In [7]:
import os
import numpy as np
import pandas as pd

# Simulate one raw GBM-driven OHLCV stock
We are simulating one stock path for the `n_steps` trading days. In particular>
* `Close`: is generated from a GBM-style process
* `Open`: comes from an overnight GBM increment
* `High`: is a positive excursion around `Open`
* `Low`: is a positive excursion around `Close`
* `Volume`: is generated separately, with larger values on larger-move days

In [8]:
def simulate_gbm_ohlcv_path(
    n_steps=10081,              # raw rows; after preprocessing -> 10080
    s0=100.0,
    mu=0.08,
    sigma=0.25,
    dt=1/252,
    overnight_scale=0.25,
    intraday_high_scale=0.6,
    intraday_low_scale=0.6,
    vol0=1_000_000.0,
    vol_sigma=0.20,
    vol_ret_coupling=6.0,
    start_date="1986-03-13",
    seed=None,
):
    rng = np.random.default_rng(seed)

    dates = pd.bdate_range(start=start_date, periods=n_steps)

    open_prices = np.zeros(n_steps, dtype=np.float64)
    high_prices = np.zeros(n_steps, dtype=np.float64)
    low_prices = np.zeros(n_steps, dtype=np.float64)
    close_prices = np.zeros(n_steps, dtype=np.float64)
    volumes = np.zeros(n_steps, dtype=np.float64)

    prev_close = s0
    log_vol_prev = np.log(vol0)

    for t in range(n_steps):
        # Overnight move: previous close -> today's open
        z_overnight = rng.normal()
        dt_overnight = dt * overnight_scale
        r_overnight = (
            (mu - 0.5 * sigma**2) * dt_overnight
            + sigma * np.sqrt(dt_overnight) * z_overnight
        )
        open_t = prev_close * np.exp(r_overnight)

        # Intraday move: open -> close
        z_intraday = rng.normal()
        dt_intraday = dt * (1.0 - overnight_scale)
        r_intraday = (
            (mu - 0.5 * sigma**2) * dt_intraday
            + sigma * np.sqrt(dt_intraday) * z_intraday
        )
        close_t = open_t * np.exp(r_intraday)

        # Intraday high/low excursions
        daily_scale = sigma * np.sqrt(dt)
        up_exc = abs(rng.normal(0.0, intraday_high_scale * daily_scale))
        down_exc = abs(rng.normal(0.0, intraday_low_scale * daily_scale))

        high_base = max(open_t, close_t)
        low_base = min(open_t, close_t)

        high_t = high_base * np.exp(up_exc)
        low_t = low_base * np.exp(-down_exc)

        # Safety
        high_t = max(high_t, open_t, close_t)
        low_t = min(low_t, open_t, close_t)

        # Volume process
        cc_ret = np.log(close_t / prev_close)
        vol_shock = rng.normal(scale=vol_sigma)
        log_vol_t = (
            0.98 * log_vol_prev
            + 0.02 * np.log(vol0)
            + vol_ret_coupling * abs(cc_ret)
            + vol_shock
        )
        volume_t = max(np.exp(log_vol_t), 1.0)

        open_prices[t] = open_t
        high_prices[t] = high_t
        low_prices[t] = low_t
        close_prices[t] = close_t
        volumes[t] = volume_t

        prev_close = close_t
        log_vol_prev = log_vol_t

    df = pd.DataFrame({
        "Date": dates,
        "Open": open_prices,
        "High": high_prices,
        "Low": low_prices,
        "Close": close_prices,
        "Volume": volumes,
    })

    return df

## Generating one fake stock
We generate one fake stock with slightly different parameters sampled uniformly in a range. Each stock has its own:
* initial price
* drift
* volatility
* average volume

The raw OHLCV data is returned **without any preprocessing** — normalization is applied separately in the window-normalization step.

In [9]:
def generate_one_fake_stock(
    stock_id,
    n_rows=10080,
    start_date="1986-01-01",
    base_seed=42,
):
    rng = np.random.default_rng(base_seed + stock_id)

    # Per-stock parameter variation
    s0    = rng.uniform(20.0, 300.0)
    mu    = rng.uniform(0.02, 0.15)
    sigma = rng.uniform(0.10, 0.45)
    vol0  = rng.uniform(1e5, 5e6)

    df_raw = simulate_gbm_ohlcv_path(
        n_steps=n_rows,
        s0=s0,
        mu=mu,
        sigma=sigma,
        dt=1/252,
        vol0=vol0,
        start_date=start_date,
        seed=base_seed + stock_id,
    )

    return df_raw

## Save raw fake stocks
Raw OHLCV CSVs (no preprocessing) are saved to `../data/fake_fts/`, one file per stock.
The normalization step below reads from this directory, so you only need to re-run that cell when changing `seq_len`.

In [10]:
def save_fake_stock_universe(
    output_dir="../data/fake_fts/",
    n_stocks=200,
    n_rows=10080,
    start_date="1986-01-01",
    base_seed=42,
):
    os.makedirs(output_dir, exist_ok=True)

    for i in range(n_stocks):
        df_raw = generate_one_fake_stock(
            stock_id=i,
            n_rows=n_rows,
            start_date=start_date,
            base_seed=base_seed,
        )

        ticker   = f"FAKE_{i+1:04d}"
        filepath = os.path.join(output_dir, f"{ticker}.csv")
        df_raw.to_csv(filepath, index=False)

    print(f"Saved {n_stocks} raw stocks ({n_rows} rows each) to {output_dir}")

# Executing it

In [11]:
save_fake_stock_universe(
    output_dir="../data/fake_fts/",
    n_stocks=200,
    n_rows=10080,       # 40 * 252
    start_date="1986-01-01",
    base_seed=42,
)

KeyboardInterrupt: 

# Window-wise normalization
For each stock in `../data/fake_fts/` we extract all sliding windows of length `seq_len` and normalize each window **column-wise** (zero mean, unit variance computed within the window).

This step is deliberately separated from data generation: changing `seq_len` only requires re-running the cell below, not re-simulating the raw paths.

Output: one `.npy` file per stock saved to `../data/fake_fts_processed/`, each with shape `(n_windows, seq_len, n_features)` where `n_features = 5` (Open, High, Low, Close, Volume).

In [14]:
import numpy as np
import pandas as pd
import os

# ── user-configurable ────────────────────────────────────────────────────────
seq_len = 64        # window length in trading days
# ─────────────────────────────────────────────────────────────────────────────

raw_dir = "../data/fake_fts/"
out_dir = "../data/fake_fts_processed/"
os.makedirs(out_dir, exist_ok=True)

cols = ["Open", "High", "Low", "Close", "Volume"]

csv_files = sorted([f for f in os.listdir(raw_dir) if f.endswith(".csv")])

for fname in csv_files:
    df = pd.read_csv(os.path.join(raw_dir, fname))
    arr   = df[cols].to_numpy(dtype=np.float32)   # (n_rows, n_features)
    dates = df["Date"].values

    n_rows, n_features = arr.shape

    if n_rows < seq_len:
        print(f"Skipping {fname}: only {n_rows} rows, need {seq_len}")
        continue

    # Truncate to a multiple of seq_len so windows tile perfectly
    n_windows   = n_rows // seq_len
    usable_rows = n_windows * seq_len

    arr_trimmed   = arr[:usable_rows]           # (n_windows * seq_len, n_features)
    dates_trimmed = dates[:usable_rows]

    # Reshape into non-overlapping windows: (n_windows, seq_len, n_features)
    windows = arr_trimmed.reshape(n_windows, seq_len, n_features)

    # Per-window, per-column normalisation
    mean = windows.mean(axis=1, keepdims=True)  # (n_windows, 1, n_features)
    std  = windows.std(axis=1,  keepdims=True)
    windows_norm = (windows - mean) / (std + 1e-8)

    # Flatten back to (n_windows * seq_len, n_features) and save as CSV
    arr_norm = windows_norm.reshape(usable_rows, n_features)

    df_out = pd.DataFrame(arr_norm, columns=cols)
    df_out.insert(0, "Date", dates_trimmed)

    out_path = os.path.join(out_dir, fname)     # same filename as raw
    df_out.to_csv(out_path, index=False)

print(
    f"Done — {len(csv_files)} stocks, "
    f"{n_windows} windows each, "
    f"seq_len={seq_len}, "
    f"total rows per file: {usable_rows}"
)

Done — 200 stocks, 157 windows each, seq_len=64, total rows per file: 10048
